# Tutorial 2: Training

This tutorial covers training the deepSCENIC model to learn gene regulatory networks from multiome data.

```{note}
This part of the tutorials takes a long time and is best run as a SLURM job from a separate python script.
```

## Setup

First, let's import the required packages and check GPU availability:

In [1]:
import torch

import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

deepSCENIC version: 0.1.0
Using device: cuda
GPU: NVIDIA A100 80GB PCIe


```{note}
Training deepSCENIC requires a GPU with at least 16-32GB VRAM due to the Enformer model. Training on CPU is technically possible but extremely slow (hours vs minutes per epoch). See the "Custom Sequence Models" section below if you need a lighter alternative.
```

## Load Preprocessed Data

We'll use the preprocessed MuData from Tutorial 1. The data must contain:
- RNA and ATAC modalities with matching cells
- TF annotations (`is_tf` column in `rna.var`)
- Region-to-gene search space (`r2g` in `uns`)
- Train/test split (`split` column in `obs` and `var`)

In [2]:
# Load preprocessed MuData from previous tutorial
mdata = ds.read("./data/MM_lines_results/preprocessed_data.h5mu")
print(
    f"Cells: {mdata.n_obs} ({(mdata.obs['split'] == 'train').sum()} train, {(mdata.obs['split'] == 'test').sum()} test)"
)
print(f"Genes: {mdata['rna'].n_vars} ({mdata['rna'].var['is_tf'].sum()} TFs)")
mdata

deepscenic.io - INFO - Loaded data/MM_lines_results/preprocessed_data.h5mu: 936 cells, 12156 genes, 281820 regions


Cells: 936 (748 train, 188 test)
Genes: 12156 (1462 TFs)


MuData object with n_obs × n_vars = 936 × 293976
  obs:	'cell_line', 'cell_state', 'split'
  uns:	'r2g'
  2 modalities
    rna:	936 x 12156
      var:	'n_cells', 'is_tf', 'chromosome', 'tss', 'split'
      uns:	'log1p', 'tf_order'
      layers:	'log_norm'
    atac:	936 x 281820
      var:	'chromosome', 'start', 'end', 'split'

## Register Genome

deepSCENIC extracts DNA sequences from genomic regions to learn TF binding patterns. Before training, you must register a genome FASTA file that matches your data's reference assembly.

```{note}
You can also use this `genome` object to fetch fasta sequences. See the API documentation.
```

In [3]:
try:
    genome = ds.get_genome()
    print(f"Genome already registered: {genome.name}")
except RuntimeError:
    ds.register_genome("./data/hg38.fa")

deepscenic.genome - INFO - Registered genome: hg38 (data/hg38.fa), chromosomes with 'chr' prefix


## Understanding Training Configuration

Training is configured through the `ds.tl.train()` function, which accepts both direct parameters and, optionally, a `ModelConfig` object. Understanding the key parameters helps you adapt training to your specific dataset and hardware.

### Two Ways to Configure Training

1. **Direct parameters** (simple): Pass common settings directly to `ds.tl.train()`
2. **ModelConfig** (advanced): Use a config object for full control over all model hyperparameters (e.g. hidden layer size)

### Key Parameters

The most important parameters to adjust:

| Parameter | Default | When to change |
|-----------|---------|----------------|
| `epochs` | 100 | Increase for better convergence |
| `batch_size` | 64 | Reduce if GPU memory errors |
| `seq_batch_size` | 1000 | Reduce if GPU memory errors |
| `lr` | 1e-4 | Lower if training unstable, increase if training too slow |

If you change either the `batch_size` or `seq_batch_size`, we recommend to change the other parameter proportionally.  
For the full parameter list, see {func}`~deepscenic.tl.train`.

```{tip}
The `seq_batch_size` (default: 1000) controls how many DNA sequences are processed per batch. This can be much larger than `batch_size` since sequences are processed independently.
```

## Train the Model

Now we're ready to train. The `ds.tl.train()` function initializes all model components, computes the initial E1 matrix from DNA sequences, and runs the training loop.

```{note}
The first training run downloads the pretrained Enformer model (~1GB). Subsequent runs use the cached version. Initial E1 computation also takes several minutes as it processes all genomic sequences.
```

In [4]:
# Train the deepSCENIC model
# For demonstration we use 100 epochs; monitor your loss curves and keep training until the model converges
model = ds.tl.train(
    mdata,
    epochs=100,
    batch_size=16,
    seq_batch_size=250,
    lr=1e-4,
    device=device,
    checkpoint_dir="./data/MM_lines_results/checkpoints/first_phase/", # save best.pt checkpoints during training
    logger = "wandb", # or 'dict', 'tensorboard'
    resume_from=None, # set this path to a checkpoint to resume training from an earlier run
)
print("Training complete!")

deepscenic.tl - INFO - Using device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/VIB.LOCAL/lukas.mahieu/.netrc.
wandb: Currently logged in as: lukas-mahieu (lukas-mahieu-vib) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


deepscenic.tl - INFO - Using 11691/12156 genes for reconstruction (train + 'both (TFs)' splits)
deepscenic.tl - INFO - Using 273545/281820 regions for ATAC reconstruction (train split)
/home/VIB.LOCAL/lukas.mahieu/.local/share/hatch/env/virtual/deepscenic/PmM2ebcu/docs/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
deepscenic.tl - INFO - Test chromosome evaluation: 8275 test regions


Init E1 (train):   0%|          | 0/1095 [00:00<?, ?it/s]

Init E1 (test):   0%|          | 0/34 [00:00<?, ?it/s]

deepscenic.tl - INFO - Starting training: 100 epochs, 936 cells, 12156 genes, 1462 TFs, 281820 regions
deepscenic.tl - INFO - Best checkpoint saving enabled: saving to 'best.pt' on improvement


...

Epoch 99/100:   0%|          | 0/47 [00:00<?, ?it/s]

Epoch 100/100:   0%|          | 0/47 [00:00<?, ?it/s]

train/atac_recon,███████▇▇▇▇▆▆▆▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train/e1_l1,█▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/e2_l1,▆███▇▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
train/kl_div,█▇▅▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂
train/rna_recon,██▇▇▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/total,██▇▆▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_cells/atac_recon,██████████▇▇▇▇▇▇▇▆▆▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁
val_cells/e1_l1,█▇▇▆▅▄▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_cells/e2_l1,█████▂▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
val_cells/kl_div,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
+5,...


deepscenic.tl - INFO - Training completed


Training complete!


## Fine-tuning E2 Matrix (Strongly Recommended)

After main training, you should fine-tune the E2 (region→gene) matrix to improve generalization. 

- **Phase 2** (`cell_split="test"`): Fine-tune E2 on held-out cells with training features. This validates that the learned regulatory relationships generalize to unseen cells.
- **Phase 3** (`feature_split="test"`): Fine-tune E2 on training cells with held-out chromosomes. This validates generalization to unseen genomic regions.

Since we're only fine-tuning the E2 matrix, these phases are very fast and should only take a couple of seconds per epoch.  

See {func}`~deepscenic.tl.finetune_e2` for full parameter reference.

In [5]:
# Phase 2: Fine-tune E2 on test cells
model = ds.tl.finetune_e2(
    model,
    mdata,
    cell_split="test",      # Use held-out cells
    feature_split="train",  # Use training features
    epochs=300,
    batch_size=64,
    reinit_e2=True,  # only reinits the E2 links for this phase's split
    checkpoint_dir="./data/MM_lines_results/checkpoints/second_phase/",
    logger = "wandb",
    logger_kwargs = {
        "project": "deepscenic",
        "name": "phase2",
    }
)

# Phase 3: Fine-tune E2 on test chromosomes
model = ds.tl.finetune_e2(
    model,
    mdata,
    cell_split="train",     # Use training cells
    feature_split="test",   # Use held-out chromosomes
    epochs=300,
    batch_size=64,
    reinit_e2=True, # only reinits the E2 links for this phase's split
    checkpoint_dir="./data/MM_lines_results/checkpoints/third_phase/",
    logger = "wandb",
    logger_kwargs = {
        "project": "deepscenic",
        "name": "phase3",
    }
)

deepscenic.tl - INFO - Using device: cuda
wandb: Currently logged in as: lukas-mahieu (lukas-mahieu-vib) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


deepscenic.tl - INFO - Using 11691/12156 genes for loss (feature_split='train')
deepscenic.tl - INFO - Using 3089422/3253274 E2 links for training (feature_split='train')
deepscenic.tl - INFO - E2 finetuning: cells=test, features=train
deepscenic.tl - INFO - Best checkpoint saving enabled: saving to 'finetune_best.pt' on improvement
deepscenic.tl - INFO - Starting E2 finetuning: 300 epochs
Finetune E2 1/300: 100%|██████████| 3/3 [00:03<00:00,  1.07s/it, loss=0.501]
deepscenic.tl - INFO - New best val loss: 0.231636 - saved to data/MM_lines_results/checkpoints/second_phase/finetune_best.pt
Finetune E2 3/300: 100%|██████████| 3/3 [00:00<00:00,  7.42it/s, loss=0.223]
deepscenic.tl - INFO - New best val loss: 0.215590 - saved to data/MM_lines_results/checkpoints/second_phase/finetune_best.pt
Finetune E2 4/300: 100%|██████████| 3/3 [00:00<00:00,  7.75it/s, loss=0.204]
deepscenic.tl - INFO - New best val loss: 0.187542 - saved to data/MM_lines_results/checkpoints/second_phase/finetune_best.p

train/e2_l1,▁▆▇▇▆▆▆▆▆▇███▇▇▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
train/rna_recon,█▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/total,█▅▃▄▃▄▄▄▄▄▃▂▂▂▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_cells/total,▇▆▆▆▅▇▆█▇▇█▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▃▂▂▂▂▁▁▁▁▁▂▂
train/e2_l1,0.00102
train/rna_recon,0.09237
train/total,0.09339
val_cells/total,0.09109


deepscenic.tl - INFO - E2 finetuning completed
deepscenic.tl - INFO - Using device: cuda


deepscenic.tl - INFO - Using 465/12156 genes for loss (feature_split='test')
deepscenic.tl - INFO - Using 143656/3253274 E2 links for training (feature_split='test')
deepscenic.tl - INFO - E2 finetuning: cells=train, features=test
deepscenic.tl - INFO - Best checkpoint saving enabled: saving to 'finetune_best.pt' on improvement
deepscenic.tl - INFO - Starting E2 finetuning: 300 epochs
Finetune E2 1/300: 100%|██████████| 12/12 [00:01<00:00,  7.24it/s, loss=0.191]
deepscenic.tl - INFO - New best val loss: 0.183989 - saved to data/MM_lines_results/checkpoints/third_phase/finetune_best.pt
Finetune E2 2/300: 100%|██████████| 12/12 [00:01<00:00,  7.31it/s, loss=0.133]
deepscenic.tl - INFO - New best val loss: 0.120520 - saved to data/MM_lines_results/checkpoints/third_phase/finetune_best.pt
Finetune E2 3/300: 100%|██████████| 12/12 [00:01<00:00,  7.58it/s, loss=0.109]
deepscenic.tl - INFO - New best val loss: 0.105287 - saved to data/MM_lines_results/checkpoints/third_phase/finetune_best.pt


train/e2_l1,▅▃▃▆▇▇▇███▅▅▃▃▃▃▃▂▂▂▃▂▂▂▂▂▂▂▂▂▂▂▂▃▂▂▂▂▂▁
train/rna_recon,█▅▄▆▆▇▇▇▇▇▅▄▄▄▃▃▃▃▃▃▂▃▂▃▃▃▂▃▃▃▂▂▂▂▂▃▃▂▂▁
train/total,▅▅▆▆▇▇███▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁
val_cells/total,█▂▃▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/e2_l1,0.00069
train/rna_recon,0.09372
train/total,0.09441
val_cells/total,0.08984


deepscenic.tl - INFO - E2 finetuning completed


## Monitor Training Progress

deepSCENIC supports logging backends for tracking training metrics:

```python
# TensorBoard logging
model = ds.tl.train(
    mdata,
    epochs=100,
    logger="tensorboard",
    log_dir="./runs/deepscenic_experiment",
    device=device,
)
```

Then launch TensorBoard in a separate terminal:
```bash
tensorboard --logdir ./runs
```

Alternatively, use Weights & Biases with `logger="wandb"`. You can set optional logger kwargs with the `logger_kwargs` argument (e.g. `logger_kwargs={project="deepscenic"}`) which will be passed to the `wandb.init()`.

## Save and Load Models

Trained models can be saved for later use. The saved file includes:
- All model weights (VAE, TF2rNet, sequence model)
- The E1 matrix cache
- Training configuration
- Feature names (TFs, genes, regions)

```{note}
Model files are large (~1GB+) when using Enformer. Models trained with custom sequence models will be smaller.
Also, checkpoints follow the same format as models saved explicitly with `model.save`, so you can load these in the exact same way.
```

In [7]:
# Save the trained model from the last epoch of phase 3
model.save("./data/MM_lines_results/checkpoints/deepscenic_model.pt")

deepscenic.tl - INFO - Saved model to ./data/MM_lines_results/checkpoints/deepscenic_model.pt


In [9]:
# Load a saved model in future sessions
model = ds.tl.load_model("./data/MM_lines_results/checkpoints/deepscenic_model.pt", device=device)

/data/groups/vib.ai/stein.aerts/lmahieu/software/miniconda3/envs/deepscenic-dev/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
deepscenic.tl - INFO - Loaded model from ./data/MM_lines_results/checkpoints/deepscenic_model.pt: 12156 genes, 1462 TFs, 281820 regions


## Custom Sequence Models

By default, deepSCENIC uses Enformer as its sequence encoder. However, you can provide your own PyTorch module instead. This is useful when:

- You want a lighter model that doesn't require 16GB+ VRAM
- You have a custom architecture tailored to your genomic context
- You want to experiment with different sequence representations

### Model Requirements

Your custom sequence model must satisfy these requirements:

| Requirement | Details |
|-------------|---------|
| **Input shape** | `(batch, seq_len, 4)` - one-hot encoded DNA sequences |
| **Output shape** | `(batch, bottleneck_size * emb_len)` - flattened embeddings |
| **Config sync** | Set matching `seq_len`, `bottleneck_size`, `emb_len` in config |

The three key parameters control the model's interface:

- **`seq_len`**: Input DNA sequence length in base pairs. Default is 640 for Enformer compatibility. Set this to match your model's expected input length.
- **`bottleneck_size`**: Embedding dimension per position. Default is 3072 (Enformer's embedding size). This is the number of features your model outputs per position.
- **`emb_len`**: Number of output positions. Default is 5 (Enformer's target_length). Combined with bottleneck_size, this determines your model's total output: `bottleneck_size * emb_len`.

In [ ]:
import torch.nn as nn

from deepscenic.tl import TrainingConfig


# Example: A lightweight convolutional sequence model
class LightweightSequenceModel(nn.Module):
    """Lightweight sequence encoder for reduced GPU memory."""

    def __init__(self, seq_len=640, bottleneck_size=256, emb_len=60):
        super().__init__()
        # Store dimensions for verification
        self.seq_len = seq_len
        self.bottleneck_size = bottleneck_size
        self.emb_len = emb_len
        self.output_dim = bottleneck_size * emb_len

        # Simple conv architecture
        self.conv = nn.Sequential(
            nn.Conv1d(4, 64, kernel_size=15, padding=7),
            nn.ReLU(),
            nn.Conv1d(64, bottleneck_size, kernel_size=15, padding=7),
            nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(emb_len)

    def forward(self, x):
        """Transform one-hot DNA to flattened embeddings."""
        # x: (batch, seq_len, 4)
        x = x.transpose(1, 2)  # (batch, 4, seq_len)
        x = self.conv(x)  # (batch, bottleneck_size, seq_len)
        x = self.pool(x)  # (batch, bottleneck_size, emb_len)
        return x.flatten(1)  # (batch, bottleneck_size * emb_len)


# Create model instance
custom_model = LightweightSequenceModel(
    seq_len=640,
    bottleneck_size=256,
    emb_len=60,
)

# Configure training with matching dimensions
config = TrainingConfig(
    sequence_model=custom_model,
    seq_len=640,  # Must match model's expected input
    bottleneck_size=256,  # Must match model's output channels
    emb_len=60,  # Must match model's output length. Usually this is =1. Enformer uses 5 (5 bins of 128bp).
    epochs=100,
)

print(f"Custom model output: {custom_model.output_dim} = {config.bottleneck_size} x {config.emb_len}")

### Training with Custom Models

```python
# Train with custom sequence model
model = ds.tl.train(mdata, config=config, device=device)

# Save the model
model.save("custom_model.pt")
```

### Loading Custom Models

When loading a model trained with a custom sequence model, you must provide an instance of the same architecture:

```python
# Recreate the model architecture (same as during training)
custom_model = LightweightSequenceModel(seq_len=640, bottleneck_size=256, emb_len=60)

# Load with the custom model
loaded_model = ds.tl.load_model(
    "custom_model.pt",
    device=device,
    sequence_model=custom_model,  # Required for custom models
)
```

```{warning}
The model architecture must exactly match what was used during training. Only the weights are saved in the checkpoint file, not the model class definition.
```

## Next Steps

Now that you have a trained model, you can:

- Diagnose model training and inspect results: {doc}`03_model_diagnosis`
- Extract and analyze the learned GRN: {doc}`04_grn_analysis`
- Simulate perturbation experiments: {doc}`05_perturbation_analysis`